# SkillOpt prompt-optimization spike (Phase 1, T002–T005)

Spec: [`spec.md`](../specs/skillopt-mlflow-integration/spec.md) ·
Plan: [`plan.md`](../specs/skillopt-mlflow-integration/plan.md) ·
Tasks: [`tasks.md`](../specs/skillopt-mlflow-integration/tasks.md) ·
**Findings → [`spike-findings.md`](../specs/skillopt-mlflow-integration/spike-findings.md)**

Goal of this notebook: pin the SkillOpt API surface before building the real
`SkillOptPromptOptimizer` (Phase 3). It confirms the in-process
`skillopt.engine.trainer.ReflACTTrainer(cfg, adapter).train()` path (R1) and the
three integration seams:

1. **Seam 1** — per-role OpenAI-compatible endpoints (`optimizer`/`target`), T003.
2. **Seam 2** — `EnvAdapter.rollout` owns the task model (project litellm `predict_fn`)
   and the shared FLEX judge; `hard`/`soft` drive both the reflection partition and the
   validation gate, T004.
3. **Seam 3** — the success-reflection toggle and the `history.json` val-score series,
   T005.

Every claim recorded in `spike-findings.md` is grounded in the installed source
(`skillopt==0.1.0`) and, where possible, run-confirmed by the cells below. The full
`train()` run (the last section) needs live `blablador`/`kisski` endpoints + API keys
in `.env`; it is gated on those being present.


## T001 — resolved dependency version

`skillopt` was added to `[project.dependencies]` in `pyproject.toml` and resolved on
Python 3.13 alongside `gepa` / `textgrad` / `mlflow` / `litellm` via `uv sync`. Its base
deps are light (`openai`, `pyyaml`, `numpy`, `openpyxl`, `azure-identity`, `azure-core`,
`httpx`) and `requires-python>=3.10`, so there was **no version conflict** — no pin/isolation
needed beyond `skillopt>=0.1.0` (only `0.1.0` is published).


In [ ]:
import sys
import skillopt

print("python  :", sys.version.split()[0])
print("skillopt:", skillopt.__version__)
assert skillopt.__version__ == "0.1.0", "spike pinned against skillopt 0.1.0"


## Setup — project imports, env, dataset, seeded split

Reuse the *exact* project building blocks the GEPA/TextGrad runs use: the dataset loader,
the seeded group-aware `split_dataset`, the shared FLEX judge, and the endpoint resolution.
This keeps the spike on the same data + judge axis as the real techniques.


In [ ]:
import json as _json
import os
from pathlib import Path

from dotenv import load_dotenv

load_dotenv()  # blablador/kisski API_BASE/API_KEY env vars

from water_assistant_agent.text2sql.core import (
    SYSTEM_PROMPT_TEMPLATE,
    USER_PROMPT_TEMPLATE,
    clean_sql,
    format_schema_for_prompt,
    load_schema,
)
from experiments.text2sql.harness import (
    ENDPOINTS,
    build_completion_kwargs,
    build_sql_judge_scorer,
    completion_with_retry,
    load_dataset,
    render_system_prompt,
)
from experiments.text2sql.sampler import split_dataset

# --- spike knobs (kept tiny: each epoch rolls out the full train split + a val gate) ---
QUESTIONS_PATH = "data/text2sql/deflated_75_sqls_prod.json"
SCHEMA_PATH = "src/water_assistant_agent/tenants/green_roof/sensordata.py"
DB_PATH = "data/water.duckdb"
SAMPLER_SEED = 42

TASK_MODEL, TASK_ENDPOINT = "openai/alias-eve", "blablador"
JUDGE_MODEL, JUDGE_ENDPOINT = "openai/qwen3.6-35b-a3b", "kisski"  # qwen36-35b-kisski
# Optimizer = SkillOpt's reflection/edit model, on SkillOpt's own model layer (seam 1).
# Model id goes straight to the OpenAI-compatible endpoint, so NO openai/ prefix.
OPTIMIZER_MODEL, OPTIMIZER_ENDPOINT = "glm-4.7", "kisski"

N_TRAIN, N_VAL = 4, 3  # a handful of records for the spike

schema_text = format_schema_for_prompt(load_schema(SCHEMA_PATH))
data = load_dataset(QUESTIONS_PATH, use_prod_questions=True)
train_set, val_set, test_set = split_dataset(
    data, SAMPLER_SEED, cache_path=Path(QUESTIONS_PATH).with_name("question_embeddings.npz")
)
train_records = train_set[:N_TRAIN]
val_records = val_set[:N_VAL]
print(f"full split: train={len(train_set)} val={len(val_set)} test={len(test_set)}")
print(f"spike subset: train={len(train_records)} val={len(val_records)}")


## Packaging gap (the #1 blocker the spike surfaced)

`skillopt==0.1.0`'s wheel **ships no prompt `.md` files**. The reflection / merge / ranking
prompts live in the GitHub repo under `skillopt/prompts/*.md`, but the repo's setuptools
build declares no `package-data`/MANIFEST for `*.md`, so neither the PyPI wheel **nor a
`git+` install** includes them. Consequence: the default `EnvAdapter.reflect` →
`run_minibatch_reflect` → `load_prompt("analyst_error")` raises `FileNotFoundError`, and the
aggregate/select stages need `merge_*`/`ranking` too.

**Workaround (non-destructive):** seed `skillopt.prompts._cache[<generic_path>] = content`
so `load_prompt(name)` returns the vendored text without the file existing on disk. The
cell below fetches the generic prompts from the pinned repo commit. Phase 3 should vendor
these `.md` files into the repo instead of fetching at runtime (recorded in findings).


In [ ]:
import urllib.request

import skillopt.prompts as _P

# Generic prompts the patch-update loop needs (failure+success reflection,
# hierarchical merge, edit ranking). meta_skill / slow_update / lr_autonomous are
# only needed when those (optional) features are enabled; we keep them off in the
# spike cfg, but vendor them too so any cfg works.
PROMPT_NAMES = [
    "analyst_error", "analyst_success",
    "merge_failure", "merge_success", "merge_final",
    "ranking",
    "meta_skill", "slow_update", "rewrite_skill", "lr_autonomous",
]
PIN = "6940e46f4e0e537a1d7ca8432f24248d0d3550f5"
_BASE = f"https://raw.githubusercontent.com/microsoft/SkillOpt/{PIN}/skillopt/prompts"
_prompts_dir = os.path.dirname(os.path.abspath(_P.__file__))


def vendor_prompt(name: str) -> None:
    """Fetch a generic prompt from the pinned commit and seed load_prompt's cache."""
    generic_path = os.path.join(_prompts_dir, f"{name}.md")
    if generic_path in _P._cache:
        return
    with urllib.request.urlopen(f"{_BASE}/{name}.md") as r:
        _P._cache[generic_path] = r.read().decode("utf-8")


for _n in PROMPT_NAMES:
    vendor_prompt(_n)

# sanity: the previously-failing load now returns text
print("analyst_error chars:", len(_P.load_prompt("analyst_error")))
print("merge_final chars  :", len(_P.load_prompt("merge_final")))
print("ranking chars      :", len(_P.load_prompt("ranking")))


## Seam 2 (T004) — `Text2SqlEnvAdapter`: rollout owns the task model + shared judge

The adapter subclasses `skillopt.envs.base.EnvAdapter`. Confirmed contract from source:

- The trainer calls `adapter.rollout(env_manager, skill_content, out_dir, use_eval_feedback=True)`
  on **train** batches and `adapter.rollout(sel_env, candidate_skill, out_dir)` on the
  **validation/selection** set (baseline + per-step gate). So one `rollout` drives both the
  reflection partition **and** the val gate — exactly what we want.
- `compute_score(results)` averages `hard`/`soft`; the gate metric defaults to `hard`
  (`select_gate_score(..., "hard")`). Setting `hard = judge verdict` makes the gated/kept
  metric the **same judge pass-rate** our `eval_fn` computes (F-001, single axis).
- The default `reflect` reads `self.analyst_workers / failure_only / minibatch_size /
  edit_budget` as **attributes** → we set them in `setup(cfg)`.
- Reflection reads each item's trajectory from `<prediction_dir>/<id>/conversation.json`
  (`fmt_minibatch_trajectories`). So `rollout` must **persist a conversation file** per
  item — returning `{id, hard, soft}` alone is not enough. It also reads optional
  `task_description`, `task_type`, `fail_reason`, `reference_text` off the result dict.
- The wheel ships no env prompts, so we override `get_error_minibatch_prompt` /
  `get_success_minibatch_prompt` to return the vendored generic analyst prompts.

EC4 guard: on a judge error we **raise** — never default a grade.


In [ ]:
import numpy as np
from skillopt.envs.base import EnvAdapter
from skillopt.gradient.reflect import run_minibatch_reflect

SCHEMA_MARKER = "Schema:"  # Phase 2 lifts this into prompt_skill.py


def _instruction_block(template: str) -> str:
    return template.split(SCHEMA_MARKER)[0].rstrip()


def _recombine(instruction: str) -> str:
    return f"{instruction}\n\n{SCHEMA_MARKER}\n\n{{schema}}\n"


class Text2SqlEnvAdapter(EnvAdapter):
    """Throwaway spike adapter: task model + judge run through the PROJECT's paths;
    only the optimizer (reflection/edit) model runs on SkillOpt's model layer."""

    def __init__(self, train_records, val_records, *, task_model, task_endpoint,
                 judge_scorer, schema_text):
        self._train = list(train_records)
        self._val = list(val_records)
        self._task_kwargs = build_completion_kwargs(task_model, task_endpoint)
        self._judge = judge_scorer
        self._schema_text = schema_text

    # --- one-time init: expose the reflect knobs the default reflect() reads ---
    def setup(self, cfg: dict) -> None:
        super().setup(cfg)
        self.minibatch_size = int(cfg["minibatch_size"])
        self.edit_budget = int(cfg["edit_budget"])
        self.analyst_workers = int(cfg.get("analyst_workers", 2))
        self.failure_only = bool(cfg.get("failure_only", True))

    def get_task_types(self):
        return ["text2sql"]

    # env_manager is just the record list for this split (supports len()).
    def build_train_env(self, batch_size, seed, **kw):
        return self._train

    def build_eval_env(self, env_num, split, seed, **kw):
        return self._val

    # wheel ships no analyst prompts -> serve the vendored generic ones
    def get_error_minibatch_prompt(self):
        return _P.load_prompt("analyst_error")

    def get_success_minibatch_prompt(self):
        return _P.load_prompt("analyst_success")

    # 0.1.0 marks reflect() abstract (no default body) -> implement it; mirrors the
    # built-in adapters by delegating to the shared minibatch reflect stage.
    def reflect(self, results, skill_content, out_dir, **kwargs):
        return run_minibatch_reflect(
            results=results,
            skill_content=skill_content,
            prediction_dir=kwargs.get("prediction_dir", os.path.join(out_dir, "predictions")),
            patches_dir=kwargs.get("patches_dir", os.path.join(out_dir, "patches")),
            workers=self.analyst_workers,
            failure_only=self.failure_only,
            minibatch_size=self.minibatch_size,
            edit_budget=self.edit_budget,
            random_seed=kwargs.get("random_seed"),
            error_system=self.get_error_minibatch_prompt(),
            success_system=self.get_success_minibatch_prompt(),
            step_buffer_context=kwargs.get("step_buffer_context", ""),
            update_mode=getattr(self, "_cfg", {}).get("skill_update_mode", "patch"),
        )

    def _task_sql(self, question, skill_content):
        system = render_system_prompt(_recombine(skill_content), self._schema_text)
        user = USER_PROMPT_TEMPLATE.format(question=question)
        resp = completion_with_retry(
            messages=[{"role": "system", "content": system},
                      {"role": "user", "content": user}],
            **self._task_kwargs,
        )
        return system, user, clean_sql(resp.choices[0].message.content or "")

    def rollout(self, env_manager, skill_content, out_dir, **kwargs):
        pred_dir = os.path.join(out_dir, "predictions")
        results = []
        for i, rec in enumerate(env_manager):
            rid = str(rec.get("id", i))
            question = rec["inputs"]["question"]
            ref_sql = rec["expectations"]["sql"]
            system, user, sql = self._task_sql(question, skill_content)

            fb = self._judge(
                inputs={"question": question},
                outputs={"sql": sql},
                expectations={"sql": ref_sql, "argilla_link": ""},
            )
            if getattr(fb, "error", None) is not None:  # EC4: never grade by default
                raise RuntimeError(f"shared judge failed on {rid!r}: {fb.error}")
            hard = 1.0 if fb.value else 0.0

            # reflection reads <pred_dir>/<id>/conversation.json
            item_dir = os.path.join(pred_dir, rid)
            os.makedirs(item_dir, exist_ok=True)
            with open(os.path.join(item_dir, "conversation.json"), "w") as f:
                _json.dump(
                    [{"role": "system", "content": system},
                     {"role": "user", "content": user},
                     {"role": "assistant", "content": sql}],
                    f, ensure_ascii=False, indent=2,
                )
            results.append({
                "id": rid,
                "hard": hard,
                "soft": hard,
                "task_description": question,
                "task_type": "text2sql",
                "reference_text": ref_sql,
                "fail_reason": "" if hard else (fb.rationale or ""),
                "n_turns": 1,
            })
        return results


print("Text2SqlEnvAdapter defined; rollout result shape:",
      "{id, hard, soft, task_description, task_type, reference_text, fail_reason, n_turns}")


## Seam 1 (T003) — per-role OpenAI-compatible endpoint cfg

`cfg` handed to `ReflACTTrainer` is a **flat** dict (the trainer reads `cfg["edit_budget"]`,
`cfg["out_root"]`, ... directly). Structured YAML is flattened by `skillopt.config.flatten_config`;
notably `optimizer.learning_rate → edit_budget` and `optimizer.min_learning_rate → min_edit_budget`.

For seam 1 the optimizer role is configured via these flat keys (consumed by
`configure_azure_openai` inside `train()`):

| flat cfg key | value |
|---|---|
| `optimizer_backend` | `"openai_chat"` |
| `optimizer_model` | model id (no `openai/` prefix) |
| `optimizer_azure_openai_endpoint` | the endpoint base_url (e.g. `…/v1`) |
| `optimizer_azure_openai_api_key` | the API key |
| `optimizer_azure_openai_auth_mode` | **`"openai_compatible"`** → plain `OpenAI(base_url, api_key)` |

`auth_mode="openai_compatible"` is the key: `model/azure_openai.py::_make_client` then builds
a vanilla `openai.OpenAI` client (not `AzureOpenAI`) pointed at our blablador/kisski base_url.

**Decision (OQ1, constant LR):** `lr_scheduler="constant"` + `min_edit_budget == edit_budget`,
so the recorded edit budget is one stable number. **Q1 / full-pass epoch:** with no
SkillOpt dataloader, `train_size` must be set explicitly; setting `batch_size == train_size`
and `accumulation=1` makes `steps_per_epoch = ceil(train_size/(batch_size*accumulation)) = 1`,
i.e. **one full-pass rollout per epoch** (one `history.json` row per epoch).

The task role stays entirely inside our `rollout`, so SkillOpt's `target` client is never
built — we still set `target_model`/`target_backend` because `train()` reads them.


In [ ]:
def resolve_endpoint(endpoint: str):
    base_var, key_var = ENDPOINTS[endpoint]
    base, key = os.environ.get(base_var), os.environ.get(key_var)
    if not base or not key:
        raise RuntimeError(
            f"endpoint {endpoint!r} not configured: set {base_var} and {key_var} in .env"
        )
    return base, key


def build_cfg(out_root, train_records, val_records):
    n_train = len(train_records)
    cfg = {
        # roles / backends
        "model_backend": "openai_chat",
        "optimizer_backend": "openai_chat",
        "target_backend": "openai_chat",
        "optimizer_model": OPTIMIZER_MODEL,
        "target_model": TASK_MODEL.removeprefix("openai/"),  # never built (task is in rollout)
        # seam 1: optimizer role -> project endpoint, plain openai-compatible auth
        "optimizer_azure_openai_auth_mode": "openai_compatible",
        # effort knobs (FR10) + constant-LR decision (OQ1)
        "num_epochs": 2,
        "edit_budget": 2,
        "min_edit_budget": 2,
        "lr_scheduler": "constant",
        "lr_control_mode": "fixed",
        "minibatch_size": 4,
        "merge_batch_size": 8,
        "analyst_workers": 2,
        "max_analyst_rounds": 1,
        "skill_update_mode": "patch",
        # full-pass epoch (Q1): batch_size == train_size, accumulation == 1 -> steps/epoch = 1
        "train_size": n_train,
        "batch_size": n_train,
        "accumulation": 1,
        "seed": SAMPLER_SEED,
        "split_seed": SAMPLER_SEED,
        # validation gate (FR12 / Q2): hard gate over the val split
        "use_gate": True,
        "gate_metric": "hard",
        "sel_env_num": len(val_records),
        "eval_test": False,
        # success-reflection toggle (FR11, Q3): failure_only = not reflect_on_success
        "failure_only": True,          # default OFF: failure reflection only
        # keep optional feature prompts out of the critical path for the spike
        "use_meta_skill": False,
        "use_slow_update": False,
        "longitudinal_pair_policy": "mixed",
        # required by train()
        "out_root": out_root,
        "skill_init": os.path.join(out_root, "skill_init.md"),
    }
    base, key = resolve_endpoint(OPTIMIZER_ENDPOINT)
    cfg["optimizer_azure_openai_endpoint"] = base
    cfg["optimizer_azure_openai_api_key"] = key
    return cfg


In [ ]:
# Offline confirmation of seam 1: the optimizer role resolves to a plain OpenAI client
# pointed at the given base_url (no network call, just client construction).
from skillopt.model import azure_openai as _ao
from skillopt.model import configure_azure_openai, set_optimizer_backend, set_optimizer_deployment

configure_azure_openai(
    optimizer_endpoint="https://example.org/v1",
    optimizer_api_key="sk-demo",
    optimizer_auth_mode="openai_compatible",
)
set_optimizer_backend("openai_chat")
set_optimizer_deployment("alias-demo")
_client = _ao.get_optimizer_client()
print("optimizer client :", type(_client).__name__)      # -> OpenAI
print("optimizer base_url:", str(_client.base_url))       # -> https://example.org/v1/
_ao._optimizer_client = None  # reset so the real run reconfigures cleanly


## Seam 3 (T005) — run `train()` in-process, then read the `history.json` series

Success-reflection toggle (Q3): `gradient.failure_only` is the key. `failure_only=True`
disables success reflection (our default-off CLI state); `failure_only=False` turns it on.
Failure reflection is always on.

`history.json` is a JSON **list**, one row per *step*. With the full-pass cfg above
(`steps_per_epoch=1`) **one row == one epoch** and `row["step"] == row["epoch"]`. Each row
carries `current_score`, `best_score`, `best_step`, `selection_hard`, `selection_soft`,
`action` (`accept` / `accept_new_best` / `reject` / `skip_*`). The val score is
`selection_hard` (mean judge `hard` over the val split) — the **same axis** as `eval_fn`,
so it can be logged under GEPA's `eval_score` metric names at `step=epoch` (F-001/F-003).

`best_skill.md` (re)written every step holds the best-on-val skill — what Phase 3 recombines
into the full template and registers.

The cell runs only when the optimizer + task + judge endpoints are configured in `.env`.


In [ ]:
import tempfile


def keys_present():
    try:
        resolve_endpoint(OPTIMIZER_ENDPOINT)
        resolve_endpoint(TASK_ENDPOINT)
        resolve_endpoint(JUDGE_ENDPOINT)
        return True
    except RuntimeError as e:
        print("SKIP train() — ", e)
        return False


RESULT = None
OUT_ROOT = None
if keys_present():
    from skillopt.engine.trainer import ReflACTTrainer

    judge_scorer = build_sql_judge_scorer(JUDGE_MODEL, JUDGE_ENDPOINT, schema_text, DB_PATH)
    adapter = Text2SqlEnvAdapter(
        train_records, val_records,
        task_model=TASK_MODEL, task_endpoint=TASK_ENDPOINT,
        judge_scorer=judge_scorer, schema_text=schema_text,
    )

    _tmp = tempfile.TemporaryDirectory()
    OUT_ROOT = _tmp.name
    cfg = build_cfg(OUT_ROOT, train_records, val_records)
    # seed skill = the prompt's instruction block (schema is fixed context, kept out)
    with open(cfg["skill_init"], "w") as f:
        f.write(_instruction_block(SYSTEM_PROMPT_TEMPLATE))

    RESULT = ReflACTTrainer(cfg, adapter).train()
    print("train() summary keys:", sorted(RESULT)[:12])


In [ ]:
# Inspect the history series + best skill (only if train() ran above).
if OUT_ROOT:
    with open(os.path.join(OUT_ROOT, "history.json")) as f:
        history = _json.load(f)
    print(f"history rows: {len(history)}  (expect == num_epochs with full-pass cfg)")
    for row in history:
        print(
            f"  step={row['step']} epoch={row['epoch']} "
            f"action={row.get('action')} "
            f"selection_hard={row.get('selection_hard')} "
            f"current={row.get('current_score'):.4f} best={row.get('best_score'):.4f}"
        )
        assert row["step"] == row["epoch"], "row->epoch mapping broken: step != epoch"

    best_path = os.path.join(OUT_ROOT, "best_skill.md")
    print("\nbest_skill.md exists:", os.path.exists(best_path))
    if os.path.exists(best_path):
        with open(best_path) as f:
            print("best_skill.md head:\n", f.read()[:400])


In [ ]:
# F-001 agreement check: history's val score must equal a direct judge pass-rate over the
# SAME val split for the SAME skill. (Run-confirmed only when train() ran.)
if OUT_ROOT:
    best = open(os.path.join(OUT_ROOT, "best_skill.md")).read()
    # score `best` over val via the adapter's own rollout (same predict_fn + judge as eval_fn)
    check = adapter.rollout(val_records, best, os.path.join(OUT_ROOT, "f001_check"))
    hard_mean = float(np.mean([r["hard"] for r in check]))
    last_best = history[-1]["best_score"]
    print(f"direct val judge pass-rate={hard_mean:.4f}  history best_score={last_best:.4f}")
    print("axes agree (within task-model sampling noise):", abs(hard_mean - last_best) < 1e-9
          or "compare manually — temperature>0 makes rollout non-deterministic")


## Summary

All four spike questions are resolved and written up in
[`spike-findings.md`](../specs/skillopt-mlflow-integration/spike-findings.md):

- **R1 / in-process** — `ReflACTTrainer(cfg, adapter).train()` runs fully in-process; no
  subprocess fallback needed. The only blocker (missing prompt assets in the wheel) is
  worked around by seeding `load_prompt`'s cache.
- **Seam 1 (T003)** — per-role endpoints via `optimizer_azure_openai_*` + `auth_mode=
  "openai_compatible"`; run-confirmed the optimizer client is a plain `OpenAI` at our base_url.
- **Seam 2 (T004)** — `rollout` runs the task model via the project `predict_fn` and the
  shared judge; `hard` drives both the partition and the gate; rollout must persist
  `<pred_dir>/<id>/conversation.json` and the default `reflect` reads
  `self.{analyst_workers,failure_only,minibatch_size,edit_budget}`.
- **Seam 3 (T005)** — `failure_only` is the success-reflection toggle; `history.json` is one
  row per step, `step==epoch` under the full-pass cfg, val score = `selection_hard` on the
  `eval_fn` axis (F-001).
- **Q1 / OQ1** — `batch_size==train_size`,`accumulation=1` → full-pass epoch;
  `lr_scheduler="constant"` + `min_edit_budget==edit_budget` → constant edit budget;
  `use_gate=True`, `eval_test=False`.
